# week-3__homework

In [2]:
from pyspark.sql import functions as F

In [3]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

In [4]:
!ls /home/iceberg/data

devices.csv  maps.csv		matches.csv  medals_matches_players.csv
events.csv   match_details.csv	medals.csv


### Explicitly broadcast JOINs medals and maps

In [5]:
path = "/home/iceberg/data/medals.csv"
df_medals = spark.read.options(header=True).csv(path)

In [6]:
path = "/home/iceberg/data/medals_matches_players.csv"
df_medals_matches_players = spark.read.options(header=True).csv(path)

In [7]:
df_mmp_agg = df_medals_matches_players.groupBy("match_id", "medal_id").agg(F.sum("count").alias("count"))

In [8]:
path = "/home/iceberg/data/matches.csv"
df_matches = spark.read.options(header=True).csv(path)

In [9]:
path = "/home/iceberg/data/maps.csv"
df_maps = spark.read.options(header=True).csv(path)

In [46]:
# note that the broadcasted table can not be in the preserved side of the join
df_j = (
    F.broadcast(df_medals).alias("df_medals")
    .join(df_mmp_agg.alias("df_mmp_agg"), on="medal_id", how="right")
    .join(F.broadcast(df_matches).alias("df_matches"), on="match_id", how="left")
    .join(F.broadcast(df_maps).alias("df_maps"), on="mapid", how="left")
)
df_j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [mapid#628, match_id#58, medal_id#60, sprite_uri#18, sprite_left#19, sprite_top#20, sprite_sheet_width#21, sprite_sheet_height#22, sprite_width#23, sprite_height#24, classification#25, description#26, name#27, difficulty#28, count#71, is_team_game#629, playlist_id#630, game_variant_id#631, is_match_over#632, completion_date#633, match_duration#634, game_mode#635, map_variant_id#636, name#131, description#132]
   +- BroadcastHashJoin [mapid#628], [mapid#130], LeftOuter, BuildRight, false
      :- Project [match_id#58, medal_id#60, sprite_uri#18, sprite_left#19, sprite_top#20, sprite_sheet_width#21, sprite_sheet_height#22, sprite_width#23, sprite_height#24, classification#25, description#26, name#27, difficulty#28, count#71, mapid#628, is_team_game#629, playlist_id#630, game_variant_id#631, is_match_over#632, completion_date#633, match_duration#634, game_mode#635, map_variant_id#636]
      :  +- BroadcastHashJoin [match_i

In [10]:
# note that the broadcasted table can not be in the preserved side of the join
df_j = (
    df_medals.alias("df_medals")
    .join(F.broadcast(df_mmp_agg).alias("df_mmp_agg"), on="medal_id", how="left")
    .join(F.broadcast(df_matches).alias("df_matches"), on="match_id", how="left")
    .join(F.broadcast(df_maps).alias("df_maps"), on="mapid", how="left")
)
df_j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [mapid#94, match_id#58, medal_id#17, sprite_uri#18, sprite_left#19, sprite_top#20, sprite_sheet_width#21, sprite_sheet_height#22, sprite_width#23, sprite_height#24, classification#25, description#26, name#27, difficulty#28, count#71, is_team_game#95, playlist_id#96, game_variant_id#97, is_match_over#98, completion_date#99, match_duration#100, game_mode#101, map_variant_id#102, name#131, description#132]
   +- BroadcastHashJoin [mapid#94], [mapid#130], LeftOuter, BuildRight, false
      :- Project [match_id#58, medal_id#17, sprite_uri#18, sprite_left#19, sprite_top#20, sprite_sheet_width#21, sprite_sheet_height#22, sprite_width#23, sprite_height#24, classification#25, description#26, name#27, difficulty#28, count#71, mapid#94, is_team_game#95, playlist_id#96, game_variant_id#97, is_match_over#98, completion_date#99, match_duration#100, game_mode#101, map_variant_id#102]
      :  +- BroadcastHashJoin [match_id#58], [match

In [11]:
# all the joins are broadcast joins.
# the only shuffle comes from the df_mmp_agg prior groupby

### Bucket join match_details, matches, and medal_matches_players on match_id with 16 buckets

In [12]:
# Setting Storage Partitioned Joins (SPJ) related configs
spark.conf.set('spark.sql.sources.v2.bucketing.enabled','true') 
spark.conf.set('spark.sql.iceberg.planning.preserve-data-grouping','true')
spark.conf.set('spark.sql.sources.v2.bucketing.pushPartValues.enabled','true')
spark.conf.set('spark.sql.requireAllClusterKeysForCoPartition','false')
spark.conf.set('spark.sql.sources.v2.bucketing.partiallyClusteredDistribution.enabled','true')

In [13]:
path = "/home/iceberg/data/match_details.csv"
df_match_details = spark.read.options(header=True).csv(path)

path = "/home/iceberg/data/matches.csv"
df_matches = spark.read.options(header=True).csv(path)

path = "/home/iceberg/data/medals_matches_players.csv"
df_medal_matches_players = spark.read.options(header=True).csv(path)

In [14]:
(df_match_details
 .writeTo("demo.bootcamp.match_details")
 .partitionedBy(F.bucket(16, "match_id"))
 .createOrReplace()
)

26/06/08 22:02:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

In [15]:
(df_matches
 .writeTo("demo.bootcamp.matches")
 .partitionedBy(F.bucket(16, "match_id"))
 .createOrReplace()
)

In [16]:
(df_medal_matches_players
 .writeTo("demo.bootcamp.medal_matches_players")
 .partitionedBy(F.bucket(16, "match_id"))
 .createOrReplace()
)

In [17]:
df_match_details = spark.table("demo.bootcamp.match_details")
df_matches = spark.table("demo.bootcamp.matches")
df_medal_matches_players = spark.table("demo.bootcamp.medal_matches_players")

In [18]:
df_j = (df_match_details
 .join(df_matches, on="match_id", how="left")
 .join(df_medal_matches_players, on=["match_id", "player_gamertag"], how="left")
)
df_j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [match_id#555, player_gamertag#556, previous_spartan_rank#557, spartan_rank#558, previous_total_xp#559, total_xp#560, previous_csr_tier#561, previous_csr_designation#562, previous_csr#563, previous_csr_percent_to_next_tier#564, previous_csr_rank#565, current_csr_tier#566, current_csr_designation#567, current_csr#568, current_csr_percent_to_next_tier#569, current_csr_rank#570, player_rank_on_team#571, player_finished#572, player_average_life#573, player_total_kills#574, player_total_headshots#575, player_total_weapon_damage#576, player_total_shots_landed#577, player_total_melee_kills#578, ... 23 more fields]
   +- SortMergeJoin [match_id#555, player_gamertag#556], [match_id#647, player_gamertag#648], LeftOuter
      :- Sort [match_id#555 ASC NULLS FIRST, player_gamertag#556 ASC NULLS FIRST], false, 0
      :  +- Project [match_id#555, player_gamertag#556, previous_spartan_rank#557, spartan_rank#558, previous_total_xp#559

In [17]:
# note that we are reading from BatchScan (Bucket) and there is no Exchange (Shuffle)

### Aggregate the joined data frame to figure out questions like:
- Which player averages the most kills per game?
- Which playlist gets played the most?
- Which map gets played the most?
- Which map do players get the most Killing Spree medals on?

In [19]:
from pyspark.sql.types import *

In [20]:
# note that at first df_j is at the (match_id, player_gamertag, medal_id) grain

df_agg = (df_j
          .withColumn("player_total_kills", (F.col("player_total_kills")).cast("int"))
          .withColumn("medal_id_count", F.struct("medal_id", "count"))
          .groupBy("match_id", "demo.bootcamp.match_details.player_gamertag")
          .agg(
              F.first("player_total_kills").alias("player_total_kills"),
              F.first(F.col("playlist_id")).alias("playlist_id"),
              F.first(F.col("mapid")).alias("mapid"),
              F.array_agg("medal_id_count").alias("array_medal_id_count")
          )
)

In [21]:
df_agg.persist()

DataFrame[match_id: string, player_gamertag: string, player_total_kills: int, playlist_id: string, mapid: string, array_medal_id_count: array<struct<medal_id:string,count:string>>]

In [24]:
df_agg.show(5)

+--------------------+---------------+------------------+--------------------+--------------------+--------------------+
|            match_id|player_gamertag|player_total_kills|         playlist_id|               mapid|array_medal_id_count|
+--------------------+---------------+------------------+--------------------+--------------------+--------------------+
|000d9c16-68a1-48d...|       DCe Baby|                11|892189e9-d712-4bd...|cb914b9e-f206-11e...|[{3261908037, 8},...|
|000d9c16-68a1-48d...|       FatTimez|                12|892189e9-d712-4bd...|cb914b9e-f206-11e...|[{250435527, 1}, ...|
|000d9c16-68a1-48d...| Scumbag Jinkie|                11|892189e9-d712-4bd...|cb914b9e-f206-11e...|[{3261908037, 4},...|
|000d9c16-68a1-48d...|    Sniipehappy|                11|892189e9-d712-4bd...|cb914b9e-f206-11e...|[{3653057799, 1},...|
|000d9c16-68a1-48d...|       Swift MG|                16|892189e9-d712-4bd...|cb914b9e-f206-11e...|[{3261908037, 5},...|
+--------------------+----------

In [25]:
df_agg.printSchema()

root
 |-- match_id: string (nullable = true)
 |-- player_gamertag: string (nullable = true)
 |-- player_total_kills: integer (nullable = true)
 |-- playlist_id: string (nullable = true)
 |-- mapid: string (nullable = true)
 |-- array_medal_id_count: array (nullable = false)
 |    |-- element: struct (containsNull = false)
 |    |    |-- medal_id: string (nullable = true)
 |    |    |-- count: string (nullable = true)



In [26]:
#### Which player averages the most kills per game?

In [27]:
(
    df_agg
    .groupBy("player_gamertag")
    .agg(F.avg("player_total_kills").alias("avg_player_total_kills"), F.count("*").alias("c"))
    .filter(F.col("c")>1000)
    .orderBy("avg_player_total_kills", ascending=False)
).show(5)

[Stage 19:======================================>                 (11 + 5) / 16]

+---------------+----------------------+----+
|player_gamertag|avg_player_total_kills|   c|
+---------------+----------------------+----+
|       EcZachly|     10.68781954887218|3325|
|  JakeWilson801|     9.779069767441861|1290|
|    ILLICIT 117|     9.742399115533443|1809|
| Ash All Mighty|      8.47852298417483|1327|
|     HD DrAsTiC|     6.953533397870281|1033|
+---------------+----------------------+----+
only showing top 5 rows



In [28]:
#### Which playlist gets played the most?

In [29]:
# yes, each match has a single playlist_id
# df_agg.groupBy("match_id").agg(F.count_distinct("playlist_id").alias("c")).orderBy("c", ascending=False).show(3)

In [30]:
(df_agg
 .groupBy("match_id").agg(F.first("playlist_id").alias("playlist_id"))
 .groupBy("playlist_id").count()
 .orderBy("count", ascending=False)
).show(5)

+--------------------+-----+
|         playlist_id|count|
+--------------------+-----+
|f72e0ef0-7c4a-430...| 7657|
|2323b76a-db98-4e0...| 3174|
|892189e9-d712-4bd...| 1974|
|c98949ae-60a8-43d...| 1828|
|f27a65eb-2d11-496...|  682|
+--------------------+-----+
only showing top 5 rows



In [31]:
#### Which playlist gets played the most?

In [32]:
(df_agg
 .groupBy("match_id").agg(F.first("mapid").alias("mapid"))
 .groupBy("mapid").count()
 .orderBy("count", ascending=False)
).show(5)

+--------------------+-----+
|               mapid|count|
+--------------------+-----+
|c7edbf0f-f206-11e...| 7049|
|c74c9d0f-f206-11e...| 1387|
|cdb934b0-f206-11e...| 1351|
|cb914b9e-f206-11e...| 1028|
|ce1dc2de-f206-11e...|  979|
+--------------------+-----+
only showing top 5 rows



In [33]:
#### Which map do players get the most Killing Spree medals on?

In [34]:
df_medals.filter(F.col("name")=="Killing Spree").select("medal_id").show()

+----------+
|  medal_id|
+----------+
|2430242797|
+----------+



In [35]:
(
    df_agg
    .withColumn("medal_id_count", F.explode("array_medal_id_count"))
    .withColumn("count", F.col("medal_id_count.count"))
    .filter(F.col("medal_id_count.medal_id")=="2430242797")
    .groupBy("mapid")
    .agg(F.sum("count").alias("killing_spree_count"))
    .orderBy("killing_spree_count", asc=False)
).limit(5).toPandas()

,mapid,killing_spree_count
0,ce89a40f-f206-11e4-b83f-24be05e24f7e,355.0
1,c7b7baf0-f206-11e4-ae9a-24be05e24f7e,725.0
2,cc74f4e1-f206-11e4-ad66-24be05e24f7e,893.0
3,cbcea2c0-f206-11e4-8c4a-24be05e24f7e,915.0
4,ca737f8f-f206-11e4-a7e2-24be05e24f7e,1049.0


### With the aggregated data set
Try different .sortWithinPartitions to see which has the smallest data size (hint: playlists and maps are both very low cardinality)

In [36]:
import pandas as pd

In [37]:
df_agg.printSchema()

root
 |-- match_id: string (nullable = true)
 |-- player_gamertag: string (nullable = true)
 |-- player_total_kills: integer (nullable = true)
 |-- playlist_id: string (nullable = true)
 |-- mapid: string (nullable = true)
 |-- array_medal_id_count: array (nullable = false)
 |    |-- element: struct (containsNull = false)
 |    |    |-- medal_id: string (nullable = true)
 |    |    |-- count: string (nullable = true)



In [38]:
df_cd = spark.sql("""
WITH counts AS (
    SELECT
        count(distinct match_id) AS match_id,
        count(distinct player_gamertag) AS player_gamertag,
        count(distinct player_total_kills) AS player_total_kills,
        count(distinct playlist_id) AS playlist_id,
        count(distinct mapid) AS mapid
    FROM demo.bootcamp.agg
)
SELECT *
FROM counts
UNPIVOT (
    count_distinct FOR metric IN (
        match_id,
        player_gamertag,
        player_total_kills,
        playlist_id,
        mapid
    )
)
ORDER BY count_distinct DESC
""").toPandas()

In [39]:
df_cd

,metric,count_distinct
0,player_gamertag,69420
1,match_id,19050
2,player_total_kills,80
3,playlist_id,23
4,mapid,16


In [40]:
sort_cols_d = {}

In [41]:
def write_and_measure(sort_cols):
    global sort_cols_d
    (df_agg
     .sortWithinPartitions(sort_cols)
     .writeTo("demo.bootcamp.agg")
     .createOrReplace()
    )
    sort_cols_str = "__".join(sort_cols)
    sort_cols_d[sort_cols_str] = spark.sql("select sum(file_size_in_bytes) from demo.bootcamp.agg.files").collect()[0][0]

In [42]:
write_and_measure(["match_id"])
write_and_measure(["player_gamertag"])
write_and_measure(["mapid", "playlist_id"])
write_and_measure(["mapid"])

In [43]:
df_bytes = (pd
 .DataFrame
 .from_dict(sort_cols_d, orient='index', columns=['table_size_in_bytes'])
 .reset_index(names='metric')
 .sort_values("table_size_in_bytes", ascending=False)
)

In [44]:
df_bytes

,metric,table_size_in_bytes
1,player_gamertag,3124025
0,match_id,2985600
3,mapid,2949615
2,mapid__playlist_id,2929898


In [45]:
df_cd.merge(df_bytes, on='metric', how='left').sort_values("count_distinct", ascending=False)

,metric,count_distinct,table_size_in_bytes
0,player_gamertag,69420,3124025.0
1,match_id,19050,2985600.0
2,player_total_kills,80,NaN
3,playlist_id,23,NaN
4,mapid,16,2949615.0


In [ ]:
# we can see that by using sortWithinPartitions() based on the columns of lowest cardinality
# enables a more efficient compresison of the data (run-length encoding)

### DROP TABLE PURGE
Clean env

In [51]:
spark.sql("show tables in demo.bootcamp").show(truncate=False)

+---------+---------------------+-----------+
|namespace|tableName            |isTemporary|
+---------+---------------------+-----------+
|bootcamp |agg                  |false      |
|bootcamp |match_details        |false      |
|bootcamp |matches              |false      |
|bootcamp |medal_matches_players|false      |
+---------+---------------------+-----------+



In [52]:
t_types = [
    'demo.bootcamp.agg',
    'demo.bootcamp.match_details',
    'demo.bootcamp.matches', 
    'demo.bootcamp.medal_matches_players'
]

df_d = {}
for t in t_types:
    spark.sql(f"drop table if exists {t} purge")
    print(f'drop table if exists {t} purge;')

drop table if exists demo.bootcamp.agg purge;


drop table if exists demo.bootcamp.match_details purge;


drop table if exists demo.bootcamp.matches purge;


[Stage 105:===========================================>         (168 + 8) / 206]

drop table if exists demo.bootcamp.medal_matches_players purge;
